In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import pathlib
import glob
from tqdm import tqdm
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from natsort import natsorted
from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.evaluation.metrics_utils import default_jsd

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation,
    update_density_estimate_multiple_observations,
    DensityEstimate,
)
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence, valid_space_grid

from dmpe.utils.sets.shared import check_in_set, load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set
from dmpe.utils.density_estimation import build_grid, select_bandwidth

from dmpe.evaluation.metrics_utils import default_jsd, default_ae, default_mcudsa, default_ksfc, default_df
from dmpe.evaluation.experiment_utils import extract_metrics_over_timesteps, extract_metrics_over_timesteps_via_interpolation, get_experiment_ids

from dmpe.related_work.random_walk import random_walk_control_law
from dmpe.utils.sets.shared import check_in_set, load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

full_column_width = 18.2
half_column_width = 8.89

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

In [ ]:
S_xu_dict = {
    Systems.FLUID_TANK: load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_S_xu_666a769d-8c1b-4b.json"),
    Systems.PENDULUM: load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_S_xu_02430b86-ae0d-42.json"),
    Systems.CART_POLE: load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json"),
}

In [ ]:
def get_uniform_target_distribution(
    dim: int,
    points_per_dim: int,
    bandwidth: float,
    grid_extend: float,
    consider_action_distribution: bool,
    penalty_function,
    obs_dim: int,
    act_dim: int,
) -> jax.Array:
    """Get a uniform target distribution for the DMPE algorithm based on the grid parameters
    and a penalty function. Only values that are not penalized by the penalty function
    are targeted in the target distribution, but all of them are to be covered uniformly.

    Args:
        dim (int): Number of dimensions for the grid.
        points_per_dim (int): The number of grid points per dimension. Always identical for each
            dimension
        bandwidth (float): The bandwidth of the kernel density estimate
        grid_extend (float): The extent of the grid in each dimension
        consider_action_distribution (bool): A flag indicating whether the action distribution
            is to be considered
        penalty_function (Callable): The penalty function that is used to determine the
            valid grid points

    Returns:
        The target distribution as a jax.Array with shape (points_per_dim**dim, 1)

    """
    z_g = build_grid(dim, low=-grid_extend, high=grid_extend, points_per_dim=points_per_dim)

    if consider_action_distribution:
        constr_func = lambda z_g: penalty_function(z_g[..., None, :obs_dim], z_g[..., None, -act_dim:])
    else:
        constr_func = lambda z_g: penalty_function(z_g[..., None, :obs_dim], None)

    ## chunk this part. How did I implement this in the past? 
    # valid_grid_point = jax.vmap(constr_func, in_axes=0)(z_g) == 0

    n_grid_points = z_g.shape[0]
    chunk_size = 20_000
    out = []
    
    for i in tqdm(jnp.arange(0, n_grid_points, chunk_size)):
        out.append(jax.vmap(constr_func, in_axes=0)(z_g[i : min(i + chunk_size, n_grid_points)]) == 0)
    valid_grid_point = jnp.concatenate(out)
    ##
    
    constrained_data_points = z_g[jnp.where(valid_grid_point == True)]

    target_distribution = DensityEstimate.from_dataset(
        constrained_data_points[None],
        z_min=-grid_extend,
        z_max=grid_extend,
        points_per_dim=points_per_dim,
        bandwidth=bandwidth,
    )
    return target_distribution.p[0] / jnp.sum(target_distribution.p[0])

In [ ]:
fluid_tank_ids = dict(
    dmpe=[
        '2025-08-27_14-55-49',
        '2025-08-27_14-57-26',
        '2025-08-27_14-59-04',
        '2025-08-27_15-00-41',
        '2025-08-27_15-02-19',
    ],
    pm_dmpe=[
        '2025-08-28_10-31-04',
        '2025-08-28_10-36-03',
        '2025-08-28_10-41-00',
        '2025-08-28_10-45-58',
        '2025-08-28_10-50-54',
    ],
    sgoats=[
        '2025-07-23_17-05-04',
        '2025-07-23_17-25-52',
        '2025-07-23_17-46-10',
        '2025-07-23_18-04-37',
        '2025-07-23_18-25-10',
    ],
    igoats=[
        '2025-07-23_16-44-48',
        '2025-07-23_17-03-43',
        '2025-07-23_17-22-58',
        '2025-07-23_17-40-58',
        '2025-07-23_17-59-50',
    ],
    random_walk=[
        '2025-07-24_11-19-31',
        '2025-07-24_11-19-43',
        '2025-07-24_11-19-54',
        '2025-07-24_11-20-06',
        '2025-07-24_11-20-18',
    ],
)

pendulum_ids = dict(
    dmpe=[
        '2025-08-27_14-57-18',
        '2025-08-27_15-00-42',
        '2025-08-27_15-04-07',
        '2025-08-27_15-07-34',
        '2025-08-27_15-11-00',
    ],
    pm_dmpe=[
        '2025-08-28_10-35-03',
        '2025-08-28_10-44-58',
        '2025-08-28_10-54-58',
        '2025-08-28_11-04-31',
        '2025-08-28_11-14-47',
    ],
    sgoats=[
        '2025-07-23_17-04-21',
        '2025-07-23_17-22-38',
        '2025-07-23_17-40-52',
        '2025-07-23_18-00-08',
        '2025-07-23_18-17-27',
    ],
    igoats=[
        '2025-07-23_16-44-37',
        '2025-07-23_17-02-47',
        '2025-07-23_17-21-38',
        '2025-07-23_17-40-06',
        '2025-07-23_17-59-04',
    ],
    random_walk=[
        '2025-07-24_11-15-20',
        '2025-07-24_11-15-32',
        '2025-07-24_11-15-44',
        '2025-07-24_11-15-55',
        '2025-07-24_11-16-06',
    ],
)

cart_pole_ids = dict(
    dmpe=[
        '2025-08-27_15-13-55',
        '2025-08-27_15-34-19',
        '2025-08-27_15-54-39',
        '2025-08-27_16-15-02',
        '2025-08-27_16-35-27',
    ],
    pm_dmpe=[
        '2025-08-28_10-44-48',
        '2025-08-28_11-04-06',
        '2025-08-28_11-23-21',
        '2025-08-28_11-42-41',
        '2025-08-28_12-02-02',
    ],
    sgoats=[
        '2025-07-24_11-52-44',
        '2025-07-24_12-31-52',
        '2025-07-24_13-10-21',
        '2025-07-24_13-50-30',
        '2025-07-24_14-28-45',
    ],
    igoats=[
        '2025-07-23_17-34-12',
        '2025-07-23_18-35-44',
        '2025-07-23_19-39-24',
        '2025-07-23_20-44-08',
        '2025-07-23_21-51-53',
    ],
    random_walk=[
        '2025-07-24_11-19-00',
        '2025-07-24_11-19-13',
        '2025-07-24_11-19-26',
        '2025-07-24_11-19-39',
        '2025-07-24_11-19-52',
    ],
)

def get_algo(exp_id, ids):
    for algo_name, exp_ids_in_algo in ids.items():
        if exp_id in exp_ids_in_algo:
            return algo_name
    return "fail"

get_fluid_tank_algo = partial(get_algo, ids=fluid_tank_ids)
get_pendulum_algo = partial(get_algo, ids=pendulum_ids)
get_cart_pole_algo = partial(get_algo, ids=cart_pole_ids)

## Generate Data:

In [ ]:
sys_name = Systems.CART_POLE

In [ ]:
if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    points_per_dim = 50
    bandwidth = select_bandwidth(2, 2, 50, 0.3).item()
    model_class = NeuralEulerODE
    get_algo_for_exp = get_fluid_tank_algo

elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    points_per_dim = 50
    bandwidth = select_bandwidth(2, 3, 50, 0.3)
    model_class = NeuralEulerODEPendulum
    get_algo_for_exp = get_pendulum_algo

elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    points_per_dim = 20
    bandwidth = select_bandwidth(2, 5, 20, 0.1)
    model_class = NeuralEulerODECartpole
    get_algo_for_exp = get_cart_pole_algo

S_xu = S_xu_dict[sys_name]

In [ ]:
S_xu.visualize()

In [ ]:
check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
constraint_function = lambda z: jnp.logical_not(check_in_S_xu(z))

In [ ]:
obs_dim = len(env.obs_description)
act_dim = env.action_dim

model_evaluator = ModelEvaluator(
    constraint_function=constraint_function,
    gt_model=EnvWrapper(env, featurize=featurize),
    obs_dim=obs_dim,
    act_dim=act_dim,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)
# data_evaluator = DataEvaluator(
#     constraint_function=constraint_function,
#     data_dim=obs_dim + act_dim,
#     points_per_dim=points_per_dim,
# )
target_distribution_p = get_uniform_target_distribution(
    dim=len(env.obs_description)+env.action_dim,
    points_per_dim=points_per_dim,
    bandwidth=bandwidth,
    grid_extend=1.0,
    consider_action_distribution=True,
    penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu(jnp.concatenate([x, u], axis=-1))),
    obs_dim=len(env.obs_description),
    act_dim=env.action_dim,
)
jsd_performance_metric = partial(default_jsd, points_per_dim=points_per_dim, bandwidth=bandwidth, target_distribution=target_distribution_p, ca=True)

data_path = DataPaths().model_learning_cs_out / str(sys_name.name).lower() / "2step"

In [ ]:
model_errors = []
jsds = []
colors = []

color_cycle = plt.rcParams["axes.prop_cycle"]()
color_map = dict(
    pm_dmpe=next(color_cycle)["color"],
    dmpe=next(color_cycle)["color"],
    sgoats=next(color_cycle)["color"],
    fail=next(color_cycle)["color"],
    igoats=next(color_cycle)["color"],
    random_walk=next(color_cycle)["color"],
)

result_paths = glob.glob(str(data_path) + "/*.eqx")

n_results = len(result_paths)
for result_path in tqdm(result_paths, total=len(result_paths)):
    result = ModelExpDataResult.from_file(
        filename=result_path,
        model_class=model_class,
    )

    algo_name = get_algo_for_exp(result.exp_id)
    colors.append(color_map[algo_name])

    # TODO: iterate over the models to find the median one?
    model_error = model_evaluator.default_metrics["pred_comp"](
        NodeModelWrapper(result.median_model, featurize=featurize), model_evaluator.gt_model
    )[1]
    model_errors.append(model_error)

    # jsd_value = data_evaluator.get_metrics(
    #     data_points=jnp.concatenate([result.observations, result.actions], axis=-1)
    # )["jsd"]
    jsd_value = jsd_performance_metric(
        result.observations, result.actions,
    )

    jsds.append(jsd_value)

In [ ]:
results = dict(
    colors=colors,
    jsds=jsds,
    model_errors=model_errors,
)

# store the numbers
with open(DataPaths().model_learning_experiments / f"reduced_space_Lm_JSD_{str(sys_name.name).lower()}.pickle", "wb") as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
plt.scatter(jsds, model_errors)
plt.xscale("log")
plt.yscale("log")

## Combined Plot:

In [ ]:
# load data
data = []

for sys in Systems:
    print(str(sys.name).lower())
    with open(DataPaths().model_learning_experiments / f"reduced_space_Lm_JSD_{str(sys.name).lower()}.pickle", 'rb') as handle:
        data.append(pickle.load(handle))

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

color_cycle = plt.rcParams["axes.prop_cycle"]()
color_map = dict(
    pm_dmpe=next(color_cycle)["color"],
    dmpe=next(color_cycle)["color"],
    sgoats=next(color_cycle)["color"],
    fail=next(color_cycle)["color"],
    igoats=next(color_cycle)["color"],
    random_walk=next(color_cycle)["color"],
)

algo_name_mapping=dict(
    pm_dmpe="$\mathrm{PM-DMPE}$",
    dmpe="$\mathrm{DMPE}$",
    sgoats="$\mathrm{sGOATS}$",
    igoats="$\mathrm{iGOATS}$",
    random_walk="$\mathrm{random-walk}$",
)

patches = []
for name, c in color_map.items():
    if name == "fail":
        continue
    patches.append(mpatches.Patch(color=c, label=algo_name_mapping[name]))
patches

empty_lines = []
for name, c in color_map.items():
    if name == "fail":
        continue
    empty_lines.append(mlines.Line2D([], [], color=c, marker='x', linestyle='None', markersize=10))

algo_names = [
    "$\mathrm{PM-DMPE}$",
    "$\mathrm{DMPE}$",
    "$\mathrm{sGOATS}$",
    "$\mathrm{iGOATS}$",
    "$\mathrm{random-walk}$",
]

In [ ]:
import matplotlib.ticker as ticker

In [ ]:
def plot_jsd_model_prediction_relation_for_multiple_systems(data: dict):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))

    for sys_idx, sys_data in enumerate(data):
        model_errors = sys_data["model_errors"]
        jsds = sys_data["jsds"]
        colors = sys_data["colors"]
        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)


    
    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.tick_params(which='both', axis="y", direction='in')
        ax.tick_params(which='both', axis="x", direction='in')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()


    axs[0].set_xticks([6e-2, 1e-1, 2e-1], minor=True)
    axs[0].set_yticks([], minor=True)
    
    axs[1].set_xticks([1e-1], minor=False)
    axs[1].set_xticks([3e-1], [r"$3 \times 10^{-1}$"], minor=True)

    axs[2].set_xticks([2e-1, 4e-1], minor=False)
    axs[2].set_xticks([], minor=True)
    axs[2].set_yticks([], minor=True)
    
    fig.legend(
        empty_lines,
        algo_names,
        prop={'size': 8 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, 0.0),
        fancybox=True,
        shadow=False, 
        ncol=len(patches)
    )
    return fig, axs

In [ ]:
fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems(data)
plt.savefig(f"JSD_LM_all_systems_ca_True_color_coded_algos.pdf", bbox_inches='tight');

## Projection to states/actions only

In [ ]:
sys_name = Systems.CART_POLE

if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    points_per_dim = 50
    bandwidth = select_bandwidth(2, 2, 50, 0.3).item()
    model_class = NeuralEulerODE
    get_algo_for_exp = get_fluid_tank_algo

elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    points_per_dim = 50
    bandwidth = select_bandwidth(2, 3, 50, 0.3)
    model_class = NeuralEulerODEPendulum
    get_algo_for_exp = get_pendulum_algo

elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    points_per_dim = 20
    bandwidth = select_bandwidth(2, 5, 20, 0.1)
    model_class = NeuralEulerODECartpole
    get_algo_for_exp = get_cart_pole_algo

S_xu = S_xu_dict[sys_name]

if sys_name == Systems.FLUID_TANK:
    grid_only_states = S_xu.grid_unflattened[:, 0, :-1]
    mask_only_states = jnp.any(S_xu.mask_unflattened, axis=-1)
    
    S_xu_only_states = DiscretizedSet(grid_only_states.reshape(-1, 1), mask_only_states.flatten(), unflattened_shape=S_xu.unflattened_shape[:-1])
    S_xu_only_states.visualize()

elif sys_name == Systems.PENDULUM:
    grid_only_states = S_xu.grid_unflattened[:, :, 0, :-1]
    mask_only_states = jnp.any(S_xu.mask_unflattened, axis=-1)
    
    S_xu_only_states = DiscretizedSet(grid_only_states.reshape(-1, 2), mask_only_states.flatten(), unflattened_shape=S_xu.unflattened_shape[:-1])
    S_xu_only_states.visualize()

elif sys_name == Systems.CART_POLE:
    grid_only_states = S_xu.grid_unflattened[:, :, :, :, 0, :-1]
    mask_only_states = jnp.any(S_xu.mask_unflattened, axis=-1)

    S_xu_only_states = DiscretizedSet(grid_only_states.reshape(-1, 4), mask_only_states.flatten(), unflattened_shape=S_xu.unflattened_shape[:-1])
    S_xu_only_states.visualize()


S_xu_only_states

In [ ]:
check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
constraint_function = lambda z: jnp.logical_not(check_in_S_xu(z))

check_in_S_xu_only_states = partial(check_in_set, grid=S_xu_only_states.grid, mask=S_xu_only_states.mask)

In [ ]:
obs_dim = len(env.obs_description)
act_dim = env.action_dim

model_evaluator = ModelEvaluator(
    constraint_function=constraint_function,
    gt_model=EnvWrapper(env, featurize=featurize),
    obs_dim=obs_dim,
    act_dim=act_dim,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)
# data_evaluator = DataEvaluator(
#     constraint_function=constraint_function,
#     data_dim=obs_dim + act_dim,
#     points_per_dim=points_per_dim,
# )

target_distribution_p = get_uniform_target_distribution(
    dim=obs_dim,
    points_per_dim=points_per_dim,
    bandwidth=bandwidth,
    grid_extend=1.0,
    consider_action_distribution=False,
    penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu_only_states(jnp.concatenate([x], axis=-1))),
    obs_dim=obs_dim,
    act_dim=0,
)
jsd_performance_metric = partial(default_jsd, points_per_dim=points_per_dim, bandwidth=bandwidth, target_distribution=target_distribution_p, ca=False)

data_path = DataPaths().model_learning_cs_out / str(sys_name.name).lower() / "2step"

In [ ]:
model_errors = []
jsds = []
colors = []

color_cycle = plt.rcParams["axes.prop_cycle"]()
color_map = dict(
    pm_dmpe=next(color_cycle)["color"],
    dmpe=next(color_cycle)["color"],
    sgoats=next(color_cycle)["color"],
    fail=next(color_cycle)["color"],
    igoats=next(color_cycle)["color"],
    random_walk=next(color_cycle)["color"],
)

result_paths = glob.glob(str(data_path) + "/*.eqx")

n_results = len(result_paths)
for result_path in tqdm(result_paths, total=len(result_paths)):
    result = ModelExpDataResult.from_file(
        filename=result_path,
        model_class=model_class,
    )

    algo_name = get_algo_for_exp(result.exp_id)
    colors.append(color_map[algo_name])

    # TODO: iterate over the models to find the median one?
    model_error = model_evaluator.default_metrics["pred_comp"](
        NodeModelWrapper(result.median_model, featurize=featurize), model_evaluator.gt_model
    )[1]
    model_errors.append(model_error)

    # jsd_value = data_evaluator.get_metrics(
    #     data_points=jnp.concatenate([result.observations, result.actions], axis=-1)
    # )["jsd"]
    jsd_value = jsd_performance_metric(
        result.observations, result.actions,
    )

    jsds.append(jsd_value)


results = dict(
    colors=colors,
    jsds=jsds,
    model_errors=model_errors,
)

# store the numbers
with open(DataPaths().model_learning_experiments / f"reduced_space_Lm_JSD_{str(sys_name.name).lower()}_only_states.pickle", "wb") as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

---

In [ ]:
def plot_jsd_model_prediction_relation_for_multiple_systems(data: dict):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))

    for sys_idx, sys_data in enumerate(data):
        model_errors = sys_data["model_errors"]
        jsds = sys_data["jsds"]
        colors = sys_data["colors"]
        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)


    
    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.tick_params(which='both', axis="y", direction='in')
        ax.tick_params(which='both', axis="x", direction='in')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()


    axs[0].set_xticks([1e-1, 2e-1], minor=True)
    axs[0].set_yticks([], minor=True)
    
    axs[1].set_xticks([], minor=False)
    axs[1].set_xticks([5e-2, 3e-1], [r"$5 \times 10^{-2}$", r"$3 \times 10^{-1}$"], minor=True)

    axs[2].set_xticks([2e-1, 4e-1], minor=False)
    axs[2].set_xticks([], minor=True)
    axs[2].set_yticks([], minor=True)
    
    fig.legend(
        empty_lines,
        algo_names,
        prop={'size': 8 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, 0.0),
        fancybox=True,
        shadow=False, 
        ncol=len(patches)
    )
    return fig, axs

In [ ]:
# load data
data = []

for sys in Systems:
    print(str(sys.name).lower())
    with open(DataPaths().model_learning_experiments / f"reduced_space_Lm_JSD_{str(sys.name).lower()}_only_states.pickle", 'rb') as handle:
        data.append(pickle.load(handle))

fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems(data)
plt.savefig(f"JSD_LM_all_systems_ca_False_color_coded_algos.pdf", bbox_inches='tight');